# Lesson 5.2 — Policies and Actions

## Learning Objectives
- Understand how Policies connect health rule events to Actions
- Configure notification actions: email, HTTP Request (webhook), and custom
- Create an end-to-end alert workflow: health rule → policy → action
- Use the Alert & Respond REST API to configure policies programmatically

**Reference**: [Alert and Respond](https://help.splunk.com/en/appdynamics-saas/get-started/26.2.0/alert-and-respond) | [Policies](https://help.splunk.com/en/appdynamics-saas/get-started/26.2.0/alert-and-respond/policies) | [Actions](https://help.splunk.com/en/appdynamics-saas/get-started/26.2.0/alert-and-respond/actions)

## 1. The Alert & Respond Architecture

```
Health Rule ──triggers──► HEALTH RULE VIOLATION EVENT
                                    │
                                    ▼
                            POLICY evaluates the event
                            ("If health rule X fires, run action Y")
                                    │
                                    ▼
                            ACTION executes:
                            - Email notification
                            - HTTP Request (Slack, PagerDuty, Jira webhook)
                            - Diagnostic session (capture data)
                            - Remediation script
```

### The three components:

| Component | Role | Config location |
|-----------|------|----------------|
| **Health Rule** | Defines the condition | Alert & Respond → Health Rules |
| **Policy** | Connects event to action, with scope filter | Alert & Respond → Policies |
| **Action** | Defines what to do (send email, call webhook) | Alert & Respond → Actions |

You can have **one policy triggering multiple actions**, and **one action used by multiple policies**.

## 2. Creating an Action

### Types of Actions:

| Action Type | Description |
|-------------|-------------|
| **Email** | Send an email to specific addresses |
| **SMS** | Send an SMS (requires SMTP/gateway) |
| **HTTP Request** | POST/GET to a URL (webhooks: Slack, PagerDuty, Teams) |
| **Custom Email** | Email with a custom template |
| **Diagnostic Session** | Start capturing snapshots/threads automatically |
| **Thread Dump** | Capture JVM thread dumps on all nodes |
| **Heap Dump** | Capture JVM heap dumps |

### Creating an HTTP Request Action (for Slack/webhook):
1. Navigate to **Alert & Respond → Actions**
2. Click **+ Add Action**
3. Select **HTTP Request**
4. Fill in:
   - **Name**: "Slack #alerts channel"
   - **URL**: your Slack Incoming Webhook URL
   - **HTTP Method**: POST
   - **Payload Template**: JSON body with AppDynamics template variables
5. Save

**Reference**: [HTTP Request Actions and Templates](https://help.splunk.com/en/appdynamics-saas/get-started/26.2.0/alert-and-respond/actions/http-request-actions-and-templates)

In [ ]:
# Example Slack webhook payload template for AppDynamics HTTP Request action
# AppDynamics replaces ${...} template variables with actual event data

slack_payload_template = """
{
  "text": "AppDynamics Alert: ${policy.name}",
  "attachments": [
    {
      "color": "${event.healthRuleViolationEvent.severity == 'CRITICAL' ? '#FF0000' : '#FFA500'}",
      "title": "${event.healthRuleViolationEvent.healthRuleDisplayName}",
      "fields": [
        {
          "title": "Application",
          "value": "${applicationName}",
          "short": true
        },
        {
          "title": "Severity",
          "value": "${event.healthRuleViolationEvent.severity}",
          "short": true
        },
        {
          "title": "Affected Entity",
          "value": "${event.healthRuleViolationEvent.affectedEntityName}",
          "short": false
        }
      ],
      "footer": "AppDynamics | ${controllerUrl}"
    }
  ]
}
"""

print("Slack Webhook Payload Template for AppDynamics HTTP Request Action:")
print(slack_payload_template)
print()
print("AppDynamics template variables used:")
import re
variables = re.findall(r'\${([^}]+)}', slack_payload_template)
for v in variables:
    print(f"  ${{{v}}}")

## 3. Creating a Policy

### Step-by-step:
1. Navigate to **Alert & Respond → Policies**
2. Click **+ Add Policy**
3. Fill in:
   - **Name**: "Checkout Critical - Slack Alert"
   - **Enabled**: Yes
4. **Scope** — define which events trigger this policy:
   - **Health Rule Violations**: Select the specific health rule (e.g., "Checkout SLA Violation")
   - **Severity**: Critical (or Warning, or both)
   - **Affected Entities**: All Business Transactions OR specific ones
5. **Actions** — what to do:
   - Click **+ Add Action**
   - Select: "Slack #alerts channel" (the action you created)
6. **Action Suppression** (optional):
   - Minimum period between alerts (e.g., don't re-alert for 30 minutes)
   - Prevents alert flooding during sustained incidents
7. Save

### Policy Trigger Events:
| Event Type | When it fires |
|------------|---------------|
| **Health Rule Violation Started** | When health rule condition is first met |
| **Health Rule Violation Continues** | Every N minutes while condition persists |
| **Health Rule Violation Ended** | When condition resolves (auto-recovery) |

In [ ]:
# Design a complete alerting workflow for the StoreFront application

alerting_workflow = {
    "name": "StoreFront Production Alerting",
    "actions": [
        {
            "name": "PagerDuty Critical Alert",
            "type": "HTTP_REQUEST",
            "method": "POST",
            "url": "https://events.pagerduty.com/v2/enqueue",
            "use_case": "Critical health rule violations — pages on-call engineer"
        },
        {
            "name": "Slack #checkout-alerts",
            "type": "HTTP_REQUEST",
            "method": "POST",
            "url": "https://hooks.slack.com/services/xxx/yyy/zzz",
            "use_case": "Warning and critical — team visibility channel"
        },
        {
            "name": "SRE Team Email",
            "type": "EMAIL",
            "recipients": ["sre-team@example.com"],
            "use_case": "Diagnostic session completion — summary email"
        },
        {
            "name": "Auto Diagnostic Session",
            "type": "DIAGNOSTIC_SESSION",
            "duration_minutes": 10,
            "use_case": "Automatically capture snapshots when checkout goes critical"
        },
    ],
    "policies": [
        {
            "name": "Checkout CRITICAL — Page On-Call",
            "triggers": ["Checkout SLA Violation — CRITICAL starts"],
            "actions": ["PagerDuty Critical Alert", "Slack #checkout-alerts", "Auto Diagnostic Session"],
            "suppress_min": 30
        },
        {
            "name": "Checkout WARNING — Notify Team",
            "triggers": ["Checkout SLA Violation — WARNING starts"],
            "actions": ["Slack #checkout-alerts"],
            "suppress_min": 15
        },
        {
            "name": "Checkout Recovery — Auto Resolved",
            "triggers": ["Checkout SLA Violation — any severity ends"],
            "actions": ["Slack #checkout-alerts"],
            "suppress_min": 0
        },
    ]
}

print(f"Alerting Workflow: {alerting_workflow['name']}")
print()
print("Actions:")
for a in alerting_workflow['actions']:
    print(f"  [{a['type']:18}] {a['name']}")
    print(f"    Use case: {a['use_case']}")

print()
print("Policies:")
for p in alerting_workflow['policies']:
    print(f"  Policy: {p['name']}")
    print(f"    Triggers: {p['triggers']}")
    print(f"    Actions:  {p['actions']}")
    print(f"    Suppress: {p['suppress_min']} minutes between alerts")

In [ ]:
# EXERCISE 5.2: Design an alerting workflow for your application
#
# Scenario:
# - The payment service has an SLA: 99.9% success rate, < 3s response time
# - Your team uses Microsoft Teams for notifications
# - Critical issues should page the on-call engineer via PagerDuty
# - All issues should be logged in Jira automatically
#
# TASK: Design the actions and policies needed

payment_alerting = {
    "actions": [
        # TODO: Add at least 3 actions
        # {"name": ..., "type": ..., "use_case": ...}
    ],
    "policies": [
        # TODO: Add at least 2 policies (critical and recovery)
        # {"name": ..., "triggers": [...], "actions": [...], "suppress_min": ...}
    ]
}

print("Payment Service Alerting Design:")
print(f"  Actions defined : {len(payment_alerting['actions'])}")
print(f"  Policies defined: {len(payment_alerting['policies'])}")
for p in payment_alerting['policies']:
    print(f"  Policy: {p.get('name', 'unnamed')}")

## Summary

- **Policy** = the bridge between a Health Rule event and one or more Actions
- **Action types**: Email, HTTP Request (webhooks), Diagnostic Session, Thread/Heap Dump
- Configure **three policies** per health rule: violation starts (critical), violation starts (warning), violation ends (recovery)
- Use **alert suppression** to avoid flooding during sustained incidents
- HTTP Request actions integrate with Slack, PagerDuty, Microsoft Teams, Jira, and any webhook-capable system

---

**Next**: [Lesson 5.3 — Dynamic Baselines and Anomaly Detection](lesson-5.3-dynamic-baselines-anomaly.ipynb)